# 02 — چهار آزمایش مقاله (بازتولید جداول ۵–۸) — دیتاست-مستقل

فقط `DATASET` را عوض کن (`dataset1` / `dataset2`).

- **Exp1 (Table 5):** پیش‌پردازش I / I+N / I+Z × ۶ مدل × ۱۱ فیچر
- **Exp2 (Table 6):** + mRMR k∈{4,6,8,10}
- **Exp3 (Table 7):** I+N/I+Z × PCA(90/95/98%)
- **Exp4 (Table 8):** I+N/I+Z × ICA(4/5)

**`FAST=True`**: smoke grid + reduced CV. **`FAST=False`**: full paper grid + nested CV 5×4×1.

In [ ]:
# Settings — change these two lines only
DATASET = "dataset1"     # "dataset1" | "dataset2"
FAST = False              # True: smoke run | False: full run

import sys, warnings
sys.path.insert(0, "src")
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
from nested_cv import append_checkpoint, evaluate_cell
from preprocessing import dedup, load_standard

df, FEATS = load_standard(f"data/{DATASET}_standard.csv")
df = dedup(df)
X, y = df[FEATS], df["CLASS_code"]
print(f"{DATASET}: {df.shape[0]} ردیف × {len(FEATS)} فیچر | کلاس‌ها: {y.value_counts().sort_index().to_dict()}")
print("فیچرها:", FEATS)
CKPT = f"results/{DATASET}_checkpoint.csv"

In [ ]:
# Exp1 — Table 5: I / I+N / I+Z × ۶ مدل
MODELS6 = ["KNN", "SVM", "DT", "RF", "GNB", "AB"]
rows = []
for prep in ["I", "N", "Z"]:
    for m in MODELS6:
        folds, agg, std = evaluate_cell(X, y, prep, None, m, fast=FAST)
        rows.append({"prep": prep, "model": m, **agg, "aac_std": round(std, 4)})
        append_checkpoint(CKPT, DATASET, "exp1_table5", prep, "all_features", m, agg["aac"], std, agg["auc"], len(df))
        print(f"{prep:>2} {m:>4}: AAC={agg['aac']:.4f}±{std:.4f} AUC={agg['auc']:.4f}")
t5 = pd.DataFrame(rows)
print("\n=== Table 5 (Exp1) — AAC mean±std ===")
print(t5.pivot(index="model", columns="prep", values="aac").round(4).to_string())
print("\nBest MLM per prep:")
print(t5.loc[t5.groupby("prep")["aac"].idxmax(), ["prep", "model", "aac"]].to_string(index=False))
t5.to_csv(f"results/{DATASET}_table5.csv", index=False, encoding="utf-8-sig")

In [ ]:
# Exp2 — Table 6: mRMR k∈{4,6,8,10} × ۳ پیش‌پردازش × ۶ مدل
rows = []
for k in [4, 6, 8, 10]:
    if k >= len(FEATS): continue
    for prep in ["I", "N", "Z"]:
        for m in MODELS6:
            folds, agg, std = evaluate_cell(X, y, prep, {"kind": "mrmr", "k": k}, m, fast=FAST)
            rows.append({"k": k, "prep": prep, "model": m, **agg, "aac_std": round(std, 4)})
            append_checkpoint(CKPT, DATASET, "exp2_table6", prep, f"mrmr_{k}", m, agg["aac"], std, agg["auc"], len(df))
            print(f"mrmr{k} {prep:>2} {m:>4}: AAC={agg['aac']:.4f}±{std:.4f}")
t6 = pd.DataFrame(rows)
best6 = t6.loc[t6.groupby(["k", "prep"])["aac"].idxmax(), ["k", "prep", "model", "aac"]]
print("\nBest per (k, prep):")
print(best6.to_string(index=False))
t6.to_csv(f"results/{DATASET}_table6.csv", index=False, encoding="utf-8-sig")

In [ ]:
# Exp3 — Table 7: PCA(90/95/98%) × (I+N | I+Z) × ۶ مدل
rows = []
for var in [0.90, 0.95, 0.98]:
    for prep in ["N", "Z"]:
        for m in MODELS6:
            folds, agg, std = evaluate_cell(X, y, prep, {"kind": "pca", "var": var}, m, fast=FAST)
            rows.append({"var": var, "prep": prep, "model": m, **agg, "aac_std": round(std, 4)})
            append_checkpoint(CKPT, DATASET, "exp3_table7", prep, f"pca_{var}", m, agg["aac"], std, agg["auc"], len(df))
            print(f"pca{var} {prep} {m:>4}: AAC={agg['aac']:.4f}±{std:.4f}")
t7 = pd.DataFrame(rows)
best7 = t7.loc[t7.groupby(["var", "prep"])["aac"].idxmax(), ["var", "prep", "model", "aac"]]
print("\nBest per (var, prep):")
print(best7.to_string(index=False))
t7.to_csv(f"results/{DATASET}_table7.csv", index=False, encoding="utf-8-sig")

In [ ]:
# Exp4 — Table 8: ICA(4/5) × (I+N | I+Z) × ۶ مدل
rows = []
for k in [4, 5]:
    if k >= len(FEATS): continue
    for prep in ["N", "Z"]:
        for m in MODELS6:
            folds, agg, std = evaluate_cell(X, y, prep, {"kind": "ica", "k": k}, m, fast=FAST)
            rows.append({"k": k, "prep": prep, "model": m, **agg, "aac_std": round(std, 4)})
            append_checkpoint(CKPT, DATASET, "exp4_table8", prep, f"ica_{k}", m, agg["aac"], std, agg["auc"], len(df))
            print(f"ica{k} {prep} {m:>4}: AAC={agg['aac']:.4f}±{std:.4f}")
t8 = pd.DataFrame(rows)
best8 = t8.loc[t8.groupby(["k", "prep"])["aac"].idxmax(), ["k", "prep", "model", "aac"]]
print("\nBest per (k, prep):")
print(best8.to_string(index=False))
t8.to_csv(f"results/{DATASET}_table8.csv", index=False, encoding="utf-8-sig")

In [ ]:
# جمع‌بندی: بهترین سلول هر آزمایش (کاندید ورودی ensemble)
summary = pd.concat([
    t5.assign(exp="exp1_no_selection"), t6.assign(exp="exp2_mrmr"),
    t7.assign(exp="exp3_pca"), t8.assign(exp="exp4_ica"),
]).sort_values("aac", ascending=False).head(12)
print(summary.to_string(index=False))
summary.to_csv(f"results/{DATASET}_top_cells.csv", index=False, encoding="utf-8-sig")
print(f"\n✓ همه‌ی نتایج در results/{DATASET}_table[5-8].csv و checkpoint ذخیره شد.")